# Lab 4: See inside your agent with tracing
**Day 2, 09:45 to 11:00.** Goal: record every model call and tool call, then use the trace to answer
"what did the agent do, how long did it take, and what did it cost?"

Part A works offline with a local JSONL tracer. Part B sends the same traces to **Langfuse**
(free cloud account at https://cloud.langfuse.com, or self-hosted).

In [ ]:
import json, os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..")) if "__file__" in globals() else os.path.abspath("../..")
sys.path.insert(0, ROOT)

from common.bank_agent import ask, build_graph
from common.local_tracer import LocalTracer

graph = build_graph()
TRACE_FILE = "traces.jsonl"
if os.path.exists(TRACE_FILE):
    os.remove(TRACE_FILE)

## Part A, Step 1: Trace a few questions locally

In [ ]:
questions = ["What's the balance on 0241234567?", "Show me MTN bundles", "Convert 75 GBP to GHS",
             "How do I reset my PIN?", "What's the balance on 0000000000?"]
for i, q in enumerate(questions):
    tracer = LocalTracer(TRACE_FILE, trace_id=f"q{i+1}", user_id="participant-01")
    r = ask(graph, q, callbacks=[tracer])
    print(f"q{i+1}: {r['tools_used']} | {r['latency_ms']} ms | {r['tokens']} tokens")

## Part A, Step 2: Read the raw trace
Each line is one event. Notice the parent/child links: the tool call belongs to the agent run.

In [ ]:
events = [json.loads(l) for l in open(TRACE_FILE)]
print(len(events), "events")
for e in events[:6]:
    print({k: e[k] for k in ("trace_id", "type") if k in e}, "|", e.get("tool") or e.get("tool_calls") or e.get("output", "")[:80])

## Part A, Step 3: Turn the trace into metrics
This is what a dashboard does: per-trace latency, tokens, tool errors.

In [ ]:
# TODO 1: for each trace_id, compute total input_tokens + output_tokens (from llm_end events),
#         number of tool calls (tool_start events) and whether any tool output contains "error".
#         Print one line per trace. Then estimate cost with PRICE_PER_1M_TOKENS.
PRICE_PER_1M_TOKENS = 0.40  # USD, illustrative; check your provider's price page
from collections import defaultdict
summary = defaultdict(lambda: {"tokens": 0, "tool_calls": 0, "tool_error": False})
for e in events:
    s = summary[e["trace_id"]]
    if e["type"] == "llm_end":
        s["tokens"] += e.get("input_tokens", 0) + e.get("output_tokens", 0)
    elif e["type"] == "tool_start":
        s["tool_calls"] += 1
    elif e["type"] == "tool_end" and "error" in e.get("output", "").lower():
        s["tool_error"] = True
total = 0
for tid, s in summary.items():
    total += s["tokens"]
    print(tid, s)
print(f"Total tokens {total}  ->  approx USD {total / 1e6 * PRICE_PER_1M_TOKENS:.6f}")

**Discuss:** which trace shows a tool error? Did the final answer admit it, or hide it?
That question (did the agent tell the truth about its tools?) is what Lab 5 turns into an automated test.

## Part B: Send traces to Langfuse
1. Create a project at https://cloud.langfuse.com and copy the keys into `.env`:
   `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST` (e.g. https://cloud.langfuse.com).
2. Run the cell. Open Langfuse > Tracing and click a trace: you will see the same tree as Part A.

In [ ]:
if os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"):
    from langfuse import get_client
    from langfuse.langchain import CallbackHandler

    handler = CallbackHandler()
    # TODO 2: run two questions with callbacks=[handler] and metadata that groups them into one session:
    #         {"langfuse_session_id": "lab4-<your name>", "langfuse_user_id": "<your name>", "langfuse_tags": ["lab4"]}
    meta = {"langfuse_session_id": "lab4-demo", "langfuse_user_id": "participant-01", "langfuse_tags": ["lab4"]}
    for q in ["What's the balance on 0551112222?", "Convert 1000 GHS to USD"]:
        print(ask(graph, q, callbacks=[handler], metadata=meta)["answer"])
    get_client().flush()  # send buffered traces before the notebook stops
    print("Sent. Open Langfuse > Tracing.")
else:
    print("Langfuse keys not set: skipping Part B. Part A's traces.jsonl shows the same data.")

## Challenges
1. In Langfuse, add a **score** to a trace by hand (thumbs up/down). Then do it from code:
   `get_client().create_score(trace_id=..., name="helpful", value=1)`.
2. Make a tool slow (`time.sleep(2)` in `common/tools.py`) and find it in the trace timeline.
3. Privacy: phone numbers are personal data. Use Langfuse's `mask` option to redact `0\d{9}` before upload.